In [27]:
import os
import io
import re
import pandas as pd
import json

In [28]:
work_fldr = r'D:\Users\Work\OneDrive - Transport Systems Catapult\Research\PhD\Models\tripchains_from_ODs'
work_fldr = r'C:\Users\haris.ballis\OneDrive - Transport Systems Catapult\Research\PhD\Models\tripchains_from_ODs'
work_fldr = r'C:\Users\Work\Desktop\local_optimisation\tripchains_from_ODs'

data_source = 'mnd'
model_name = 'greenwich'
trips_direction = 'exclExtToExt2'
distr_contr_enabled_switch = [True, False]

# Dimensions used for the distribution comparison. List should have same order as in main function (of full script)
distr_dims = ['NumLegs', 'DistanceGroup']#, 'TimePeriodStart', 'TimePeriodEnd']

global_separator = '-'

In [29]:
def prepare_paths(generation_inputs_fldr, generation_outputs_fldr, optim_outputs_fldr):

    paths = dict()
    paths['zones_info'] = os.path.join(generation_inputs_fldr, 'zones_info.csv')
    paths['od'] = os.path.join(generation_outputs_fldr, 'od_filtered.csv')
    paths['tripchains_description'] = os.path.join(generation_outputs_fldr, 'tripchains_description.csv')
    paths['tripchains_info'] = os.path.join(generation_outputs_fldr, 'tripchains_info.csv')
    paths['target_distribution'] = os.path.join(generation_outputs_fldr, 'target_distribution.csv')
    paths['optimisation_results'] = os.path.join(optim_outputs_fldr, 'results.json')

    return paths

In [30]:
def get_optimisation_results(infilepath):
    with open(infilepath) as f:
        data = json.load(f)

        results = data['Solution'][1]['Variable']

        df_data = list()
        for chain, val in results.items():
            cid = re.search('\[(\d+)\]', chain, re.DOTALL).group(1)
            times_used = val['Value']
            df_data.append([int(cid), times_used])

        df = pd.DataFrame(df_data, columns=['TripChainId', 'TimesUsed']).set_index('TripChainId').squeeze()
        return df

def get_optimisation_results(infilepath):
    import re

    with open(infilepath) as f:
        lines = f.read()

    match = re.search(':=\n(.*);', lines, re.DOTALL)
    
    try:
        result = match.group(1)
        result = io.StringIO(result)

        df = pd.read_csv(result, names=['TripChainId', 'TimesUsed'], 
                         index_col=['TripChainId'], delim_whitespace=True,
                         squeeze=True)
        return df
    # No results returned
    except AttributeError:
        return None
    

In [31]:
def compare_distribution(tripchains_distribution, tripchains_distribution_grouping,
                         distr_dims, optimisation_results):
    
    df = tripchains_distribution_grouping.join(optimisation_results).dropna()
    df = df.groupby(distr_dims)['OptimisationTrips'].sum()
    optimisation_results_distribution = df.div(df.sum())
    optimisation_results_distribution.name = 'Optimisation'
    tripchains_distribution_check = tripchains_distribution.groupby(distr_dims).sum()
    tripchains_distribution_check.name = 'Nts'

    comp_distribution = tripchains_distribution_check.to_frame().join(optimisation_results_distribution)
    comp_distribution = comp_distribution.dropna()
    
    return comp_distribution

In [32]:
def convert_optimisation_output_to_trips(tcs_descr, optim_output):

    # Append how many times the tripchain has been used by the optimiser 
    df = tcs_descr.join(optim_output, how='right')
    
    # Create list with the OdPairsId which consist each of the used chains 
    tc_od_pairs = [x.split(global_separator) for x in df.OdPairsId.values]
    # Zip each tripchain's description with the times used 
    times_used_tc_od_pairs = list(zip(tc_od_pairs, df.TimesUsed))
    
    # create a df which contains the OdPairId and how many times it has been used
    data = list()
    for tc_od_pairs, times_used in times_used_tc_od_pairs:
        for od_pair in tc_od_pairs:
            data.append([int(od_pair), int(times_used)])

    df = pd.DataFrame(data, columns=['OdPairId', 'TimesUsed'])
    optimisation_trips = df.groupby('OdPairId').sum().squeeze()
    optimisation_trips.name = 'OptimisationTrips'
    return optimisation_trips


In [33]:
def get_optimisation_trips(paths):
    
    tcs_descr = pd.read_csv(paths['tripchains_description'], index_col=['TripChainId'])
    optim_output = get_optimisation_results(paths['optimisation_results'])
    
    if optim_output is not None:
        optimisation_trips = convert_optimisation_output_to_trips(tcs_descr, optim_output)
        return optimisation_trips, optim_output
    else:
        return None

In [34]:
def include_run_info(df, legs, trips_thresh, distr_constr_enabled):
    try:
        df = df.to_frame()
    except AttributeError:
        pass
    
    df['Legs'] = int(legs)
    df['TripsThresh'] = int(trips_thresh)
    df['DistrConstr'] = distr_constr_enabled
    df.set_index(['Legs', 'TripsThresh', 'DistrConstr'], append=True, inplace=True)
    df = df.squeeze()
    
    return df

In [35]:
def prepare_ods_comparison(optim_ods, init_ods):
    comp = init_ods.join(optim_ods)
    comp.index = comp.index.droplevel('OdPairId')
    comp.set_index(['OrigZoneId', 'DestZoneId', 'Purpose', 'TimePeriod'], append=True, inplace=True)
    comp.columns.name = 'TripsSource'
    comp = comp.stack()
    comp.name = 'Trips'
    return comp

In [36]:
def prepare_inputs_from_folder(infldr):
    fldrs = [name for name in os.listdir(infldr) if os.path.isdir(os.path.join(infldr, name))]
    inputs = []
    for fldr in fldrs:
        matches = re.findall('(\d)legs-over(\d+)trips', fldr)
        if matches:
            inputs.append(matches[0])
        
    return inputs

In [37]:
def prepare_ditribution_comparison(fldrpaths):
    tcs_descr = pd.read_csv(fldrpaths['tripchains_description'], index_col=['TripChainId'], squeeze=True)
    tcs_info = pd.read_csv(fldrpaths['tripchains_info'], index_col=['TripChainId'])
    optim_output = get_optimisation_results(fldrpaths['optimisation_results'])
    target_distr = pd.read_csv(fldrpaths['target_distribution'], index_col=distr_dims)['Percentage']

    df = tcs_info.join(optim_output, how='right')
    optim_distr = df.groupby(distr_dims)['TimesUsed'].sum()
    optim_distr = optim_distr.div(optim_distr.sum())
    optim_distr.name = 'Percentage'
    
    distr_comp = pd.concat([target_distr, optim_distr], axis=1)
    distr_comp.columns = ['Target', 'Optimisation']
    distr_comp.columns.name = 'Source'
    distr_comp = distr_comp.stack()
    distr_comp.name = 'Percentage'

    return distr_comp

In [38]:
##### MAIN
generation_inputs_fldr = os.path.join(work_fldr, r'generation\inputs')


infldr = os.path.join(work_fldr, r'generation/outputs', '{}\{}\{}'.format(data_source, model_name, trips_direction))
inputs = prepare_inputs_from_folder(infldr)

concat_optim_od = list()
concat_optim_nchains = list()
concat_init_ods = list()
concat_distr_comps = list()

for legs, trips_thresh in inputs:
    for distr_constr_enabled in distr_contr_enabled_switch:
        
        generation_outputs_fldr = (os.path.join(work_fldr, r'generation\outputs\{}\{}\{}\{}legs-over{}trips'
                                   .format(data_source, model_name, trips_direction,
                                           legs, trips_thresh)))

        optim_outputs_fldr = (os.path.join(generation_outputs_fldr, 'constrEnabled-{}' .format(distr_constr_enabled)))
        fldrpaths = prepare_paths(generation_inputs_fldr, generation_outputs_fldr, optim_outputs_fldr)

        optim_od, optim_nchains = get_optimisation_trips(fldrpaths)
        
        if optim_od is None:
            print('No results for legs:{}-trips:{}-distrConstr:{}'.format(legs, trips_thresh, distr_constr_enabled))
            continue
            
        optim_od = include_run_info(optim_od, legs, trips_thresh, distr_constr_enabled)
        optim_nchains = include_run_info(optim_nchains, legs, trips_thresh, distr_constr_enabled)
        
        concat_optim_od.append(optim_od)
        concat_optim_nchains.append(optim_nchains)
        
        idx_cols_od = ['OrigZoneId', 'DestZoneId', 'Purpose', 'TimePeriod']
        init_od = pd.read_csv(fldrpaths['od'], index_col='OdPairId')
        init_od = include_run_info(init_od, legs, trips_thresh, distr_constr_enabled)
        concat_init_ods.append(init_od)
        
        distr_comp = prepare_ditribution_comparison(fldrpaths) 
        distr_comp = include_run_info(distr_comp, legs, trips_thresh, distr_constr_enabled)
        concat_distr_comps.append(distr_comp)

optim_nchains = pd.concat(concat_optim_nchains)
optim_ods = pd.concat(concat_optim_od)
init_ods = pd.concat(concat_init_ods)
distr_comps = pd.concat(concat_distr_comps)

zones_info = pd.read_csv(fldrpaths['zones_info'], index_col=['ZoneId'])

ods_comparison = prepare_ods_comparison(optim_ods, init_ods)

#trips_comparison = trips_comparison.reorder_levels(['Legs', 'TripsThresh', 'DistrConstr', 'OdPairId'])
#trips_comparison.columns.name = 'TripsSource'
#trips_comparison = trips_comparison.stack()
#trips_comparison.name = 'Trips'
#trips_comparison.to_csv('tmp.csv', header=True)

In [39]:
df = distr_comps.copy()
df = df.xs(True, level='DistrConstr')
df.unstack('Source')

Source                                  Optimisation    Target
NumLegs DistanceGroup Legs TripsThresh                        
2       (0.0, 5.0]    3    25               0.416995  0.426984
                           50               0.417005  0.426984
                           300              0.469565  0.479068
        (10.0, inf]   3    25               0.212057  0.202086
                           50               0.212045  0.202086
                           300              0.227329  0.226736
        (5.0, 10.0]   3    25               0.252272  0.262212
                           50               0.252271  0.262212
                           300              0.303106  0.294196
3       (0.0, 5.0]    3    25               0.015109  0.025102
                           50               0.015142  0.025102
        (10.0, inf]   3    25               0.057710  0.047744
                           50               0.057711  0.047744
        (5.0, 10.0]   3    25               0.045857  0.035872
                           50               0.045826  0.035872

In [40]:
df = ods_comparison.groupby(level=['Legs', 'TripsThresh', 'DistrConstr', 'TripsSource', 'Purpose']).sum()

df = df.unstack('TripsSource')

df['Diff'] = df.DailyTrips - df.OptimisationTrips
df['Diff_perc'] = df.OptimisationTrips/df.DailyTrips - 1

df.style.format({
    'OptimisationTrips': '{:,.0f}'.format,
    'DailyTrips': '{:,.0f}'.format,
    'Diff': '{:,.0f}'.format,
    'Diff_perc': '{:.0%}'.format
})

In [42]:
df.unstack('TripsThresh')

TripsSource                       DailyTrips                    \
TripsThresh                              25       50       300   
Legs DistrConstr Purpose                                         
3    False       HB_FromHome         95072.0  77009.0  13454.0   
                 HB_ToHome           87466.0  67985.0   9287.0   
                 NHB_NonHomeBased   125923.0  99266.0  29995.0   
     True        HB_FromHome         95072.0  77009.0  13454.0   
                 HB_ToHome           87466.0  67985.0   9287.0   
                 NHB_NonHomeBased   125923.0  99266.0  29995.0   

TripsSource                       OptimisationTrips                   \
TripsThresh                                     25       50      300   
Legs DistrConstr Purpose                                               
3    False       HB_FromHome                57059.0  43643.0  5803.0   
                 HB_ToHome                  57059.0  43643.0  5803.0   
                 NHB_NonHomeBased           34991.0  22153.0     NaN   
     True        HB_FromHome                26408.0  17501.0   805.0   
                 HB_ToHome                  26408.0  17501.0   805.0   
                 NHB_NonHomeBased            3134.0   2077.0     NaN   

TripsSource                            Diff                   Diff_perc  \
TripsThresh                             25       50       300       25    
Legs DistrConstr Purpose                                                  
3    False       HB_FromHome        38013.0  33366.0   7651.0 -0.399834   
                 HB_ToHome          30407.0  24342.0   3484.0 -0.347644   
                 NHB_NonHomeBased   90932.0  77113.0      NaN -0.722124   
     True        HB_FromHome        68664.0  59508.0  12649.0 -0.722232   
                 HB_ToHome          61058.0  50484.0   8482.0 -0.698077   
                 NHB_NonHomeBased  122789.0  97189.0      NaN -0.975112   

TripsSource                                            
TripsThresh                             50        300  
Legs DistrConstr Purpose                               
3    False       HB_FromHome      -0.433274 -0.568678  
                 HB_ToHome        -0.358050 -0.375148  
                 NHB_NonHomeBased -0.776832       NaN  
     True        HB_FromHome      -0.772741 -0.940166  
                 HB_ToHome        -0.742576 -0.913320  
                 NHB_NonHomeBased -0.979076       NaN

In [ ]:
optim_nchains.groupby(level=[1,2,3]).sum()

In [ ]:
optim_trips.xs(True, level='DistrConstr')